# 00 · Pre-train a tiny GPT from scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/57Ajay/modeltune/blob/main/notebooks/00_pretrain_tiny_gpt.ipynb)

Companion to [chapter 02: Pre-training](https://github.com/57Ajay/modeltune/blob/main/course/02_pretraining.md). We train a ~12M-parameter
modern transformer (RMSNorm, RoPE, SwiGLU) on TinyStories with the plain next-token objective, the same one
used for Llama and Qwen. About 10–15 minutes on a T4 (estimate).

**Watch for:** loss starting at ≈ ln(4096) = 8.3 (random guessing), and samples going from noise → words → grammar → stories.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Any newer GPU works too; the code picks fp16 or bf16 by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Get the course code, then install the pinned libraries from pyproject.toml with uv.
# Colab already ships a CUDA build of PyTorch; `uv pip install --system` keeps it instead of downloading another.
!test -d modeltune || git clone -q https://github.com/57Ajay/modeltune.git
%cd modeltune
!git pull -q
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q -r pyproject.toml

In [ ]:
# Hugging Face token: add HF_TOKEN in Colab's key icon (Secrets) panel, with WRITE access to push models.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()
# Pushing to someone else's namespace fails: change hub.repo_id in the configs (or override it) to YOUR username.

## 1. Train
Read `pretrain/tiny_gpt.py` alongside: it's ~300 lines and every block maps to a section of chapter 02.
Add `--push_repo <you>/tiny-gpt-tinystories` to upload the result to the Hub.

In [ ]:
!python pretrain/tiny_gpt.py --out_dir outputs/tiny-gpt

## 2. Plot the loss curve

In [ ]:
import json, matplotlib.pyplot as plt
log = json.load(open("outputs/tiny-gpt/training_log.json"))
steps = [r["step"] for r in log]
plt.plot(steps, [r["train_loss"] for r in log], label="train")
plt.plot(steps, [r["val_loss"] for r in log], label="val")
plt.xlabel("step"); plt.ylabel("cross-entropy (nats)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 3. Sample with different temperatures
Low temperature = safe and repetitive; high = creative and incoherent. This is *decoding*, not training.

In [ ]:
import sys, torch
sys.path.insert(0, "pretrain")
from tiny_gpt import TinyGPT
from tokenizers import Tokenizer
ckpt = torch.load("outputs/tiny-gpt/model.pt", map_location="cuda")
model = TinyGPT(**ckpt["config"]).cuda().eval(); model.load_state_dict(ckpt["model"])
tok = Tokenizer.from_file("outputs/tiny-gpt/tokenizer.json")
idx = torch.tensor([tok.encode("Once upon a time, a little dragon").ids], device="cuda")
for t in (0.3, 0.8, 1.5):
    out = model.generate(idx, 80, temperature=t, eot=tok.token_to_id("<|endoftext|>"))
    print(f"--- temperature {t}:\n{tok.decode(out[0].tolist())}\n")

## Exercises
1. **Scaling:** run `--n_layer 2 --d_model 128` and `--n_layer 8 --d_model 512`. Compare final val loss vs parameter count.
2. **Learning rate:** try `--lr 3e-3` and `--lr 1e-4`. What does too-high look like? Too-low?
3. **Data vs steps:** `--num_stories 20000` (repeats data many times) vs the default. Can you spot overfitting (train ↓, val ↑)?
4. **Tokenizer:** `--vocab_size 512` vs `8192`. Fewer tokens per story or more? How does that change loss *per token*?